In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np      
from scipy.integrate import complex_ode

from tasks import make_static_sin_task
from encoding_masking import build_masked_input, compute_tdm_params
from Reservoirs.ELM_static import run_elm_static, ELMStaticParams
from readout import split_states_targets, fit_ridge_readout, predict_readout
from metrics import mse, nrmse

# scipy numerical integrator did not prove useful

In [2]:
L = 1000
N = 50

#reservoir parameters:
p = 1.11 #normalised injection current
eta = 0.08 #input strength 
alpha = 3 #linewidth enhancemenet factor
phi = 0.0 #phase shift
kappa = 10**8

task_seed = 0
mask_seed = 42

g = 1.0
ridge_alpha = 1e-4
split = 0.8

theta = 10**-10
dt = 10**-11

TDM_parameters = compute_tdm_params(N, theta, dt)

tap_stride = TDM_parameters.tap_stride
Nd_loop = TDM_parameters.Nd_loop
print(TDM_parameters)

TDMParams(N=50, theta=1e-10, dt=1e-11, T=5e-09, tau=5.0500000000000006e-09, Nd_loop=500, Nd_delay=505, tap_stride=10)


In [3]:
task = make_static_sin_task(L=L, seed=task_seed)
x, y = task.x, task.y

In [4]:
x_norm, mask, V = build_masked_input(x, N=N, rng=mask_seed)

In [5]:
def expand_virtual_nodes(V, tap_stride, Nd):

    L, N = V.shape
    #Nd = N * tap_stride

    Vs = np.zeros(L * Nd)

    for l in range(L):
        for n in range(N):
            for step in range(n * tap_stride, (n + 1) * tap_stride):
                Vs[step + l * Nd] = V[l, n]

    return Vs


In [16]:
def lk_rhs(t, y, delay_buf, di, v_slot, alpha, kappa, phi, p, eta):

    E = y[0]
   # n = y[1].real # must force to be real as solver assumes everything is complex(n is real by nature but solver turns it complex)
    n = y[1]
    
    E_delay = delay_buf[di] #internal solver will do intermediate steps between t and t+dt so consider adding intermediate steps.

    I = np.abs(E)**2
    dEdt = ((1.0 + 1j*alpha) * E) - (I * E) + (kappa * E_delay * np.exp(1j*phi))
    dndt = p + (eta * v_slot) - ((1.0 + I) * n)

    return np.array([dEdt, dndt + 0j], dtype=np.complex128)

def simulate_lk_new(Vs, dt, Nd_delay, *, alpha, kappa, phi, p, eta, E0=1e-3+0j, n0=0.0, rtol=1e-6, atol=1e-9):
   
    Vs = np.asarray(Vs, dtype=float)
    T_steps = Vs.size

    E_hist = np.zeros(T_steps, dtype=np.complex128)
    n_hist = np.zeros(T_steps, dtype=float)

    delay_buf = np.full(Nd_delay, E0, dtype=np.complex128)
    di = 0

    y0 = np.array([E0, n0 + 0j], dtype=np.complex128) # initial state of resevoir
    

#---------------------------------

    print("y0:", y0)
    print("type(y0):", type(y0))
    print("y0.shape:", y0.shape)
#-----------------------------------

    
    solver = complex_ode(lk_rhs)
    solver.set_integrator("dopri5", rtol=rtol, atol=atol, nsteps=1000)
    solver.set_initial_value(y0, t=0.0) #set initial resevoir state.



#---------------------------------------
    print("solver.y:", solver.y)
    print("type(solver.y):", type(solver.y))
    
#---------------------------------------------
    for step in range(T_steps):
        
        v_slot = float(Vs[step])
        t0 = solver.t


#----------------------------------------------------------------
        if step == 0:
            print("Before first integrate:")
            print("solver.t:", solver.t)
            print("v_slot:", v_slot, type(v_slot))
            print("di:", di, type(di))
            print("delay_buf.shape:", delay_buf.shape)
            print("delay_buf[di]:", delay_buf[di])


#-------------------------------------------------
        
        solver.set_f_params(delay_buf, di, v_slot, alpha, kappa, phi, p, eta)

        y_next = solver.integrate(t0 + dt)

        E = y_next[0]
        n = y_next[1].real

        E_hist[step] = E
        n_hist[step] = n

     
        delay_buf[di] = E
        di = (di + 1) % Nd_delay

    return E_hist, n_hist    

In [17]:
tap_stride = TDM_parameters.tap_stride  
Nd_delay = TDM_parameters.Nd_delay    
tau = TDM_parameters.tau


Vs = expand_virtual_nodes(V, tap_stride=tap_stride, Nd=Nd_loop) 

In [18]:
#debug ODE itself
test_y = np.array([1e-3 + 0j, 0.0 + 0j], dtype=np.complex128)
test_delay = np.full(Nd_delay, 1e-3 + 0j, dtype=np.complex128)

out = lk_rhs(0.0, test_y, test_delay, 0, float(Vs[0]), alpha, kappa, phi, p, eta)
print(out)
print(out.shape)

[1.00000001e+05+0.003j 1.13612008e+00+0.j   ]
(2,)


In [19]:
E_hist, n_hist = simulate_lk_new(Vs, dt, Nd_delay, alpha=alpha, kappa=kappa, phi=phi, p=p, eta=eta)

R = extract_R_from_E(E_hist, L=L, N=N, tap_stride=tap_stride, Nd_loop=Nd_loop)
print("R shape:", R.shape)  # (L, N)


y0: [0.001+0.j 0.   +0.j]
type(y0): <class 'numpy.ndarray'>
y0.shape: (2,)
solver.y: [0.001+0.j 0.   +0.j]
type(solver.y): <class 'numpy.ndarray'>
Before first integrate:
solver.t: 0.0
v_slot: 0.32650105571954696 <class 'float'>
di: 0 <class 'int'>
delay_buf.shape: (505,)
delay_buf[di]: (0.001+0j)


TypeError: 'int' object is not subscriptable